# LeetCode 1616 — Split Two Strings to Make Palindrome bằng Python

Link đề: https://leetcode.com/problems/split-two-strings-to-make-palindrome/ · Độ khó: **Medium** · Chủ đề: Two Pointers, String

Skeleton LeetCode:

```python
class Solution:
    def checkPalindromeFormation(self, a: str, b: str) -> bool:
        ...
```

## 1. Mục tiêu học

1. Kiểm tra **palindrome** (chuỗi đối xứng) bằng đảo chuỗi `s[::-1]` và bằng **hai con trỏ** trên một đoạn `s[i..j]`.
2. Hiểu cấu trúc của chuỗi ghép `a[:i] + b[i:]`: phần **vỏ ngoài** ghép cặp `a[k]` với `b[n−1−k]`, phần **lõi giữa** nằm trọn trong một chuỗi.
3. Đi từ vét cạn `O(n²)` (thử mọi điểm cắt) lên **tham lam hai con trỏ** `O(n)` và cách **mở rộng từ tâm** `O(n)`.
4. Chứng minh vì sao "khớp vỏ ngoài **càng dài càng tốt**" là đủ — nhờ tính chất **palindrome đồng tâm**.
5. Đừng quên **cả hai** cách ghép (`a_prefix + b_suffix` và `b_prefix + a_suffix`) và các điểm cắt ở **hai đầu** (tiền tố/hậu tố rỗng).

## 2. Đề bài & ví dụ

Cho hai chuỗi `a`, `b` **cùng độ dài** `n`. Chọn **một chỉ số cắt** `i` (`0 ≤ i ≤ n`) và cắt **cả hai** chuỗi tại cùng chỉ số đó:
- `a = a_prefix + a_suffix` với `a_prefix = a[:i]`, `a_suffix = a[i:]`;
- `b = b_prefix + b_suffix` với `b_prefix = b[:i]`, `b_suffix = b[i:]`.

Kiểm tra xem `a_prefix + b_suffix` **hoặc** `b_prefix + a_suffix` có phải **palindrome** không. Tiền tố hoặc hậu tố được phép **rỗng** (ví dụ `s = "abc"` có các cách cắt `"" + "abc"`, `"a" + "bc"`, `"ab" + "c"`, `"abc" + ""`).

Trả về `True` nếu tạo được palindrome, ngược lại `False`.

**Ví dụ 1:** `a = "x"`, `b = "y"` → `True`. Cắt tại `i = 0`: `a_prefix + b_suffix = "" + "y" = "y"` là palindrome. (Tổng quát: nếu `a` hoặc `b` đã là palindrome thì đáp án luôn `True`.)

**Ví dụ 2:** `a = "xbdef"`, `b = "xecab"` → `False`.

**Ví dụ 3:** `a = "ulacfd"`, `b = "jizalu"` → `True`. Cắt tại `i = 3`: `"ula" + "alu" = "ulaalu"` là palindrome.

**Ràng buộc:**
- `1 ≤ a.length, b.length ≤ 10⁵`
- `a.length == b.length`
- `a`, `b` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
import random
import time
from typing import List

examples = [
    ("x", "y", True),
    ("xbdef", "xecab", False),
    ("ulacfd", "jizalu", True),
]
for a, b, exp in examples:
    print(f"a={a!r}, b={b!r} -> kỳ vọng {exp}")

a='x', b='y' -> kỳ vọng True
a='xbdef', b='xecab' -> kỳ vọng False
a='ulacfd', b='jizalu' -> kỳ vọng True


## 3. Kiến thức nền

### 3.1 Palindrome và đảo chuỗi bằng slicing

Chuỗi `s` là palindrome nếu đọc xuôi = đọc ngược: `s == s[::-1]`. Chuỗi rỗng và chuỗi một ký tự luôn là palindrome.

In [2]:
for s in ["", "a", "abba", "abcba", "abca", "ulaalu"]:
    print(repr(s), "->", s == s[::-1])

'' -> True
'a' -> True
'abba' -> True
'abcba' -> True
'abca' -> False
'ulaalu' -> True


### 3.2 Slicing `s[i:j]` và chỉ số "gương" `n − 1 − k`

`s[i:j]` lấy các ký tự từ `i` đến `j − 1`. Trong chuỗi độ dài `n`, ký tự ở vị trí `k` có vị trí **đối xứng** (gương) là `n − 1 − k`. Palindrome ⇔ `s[k] == s[n−1−k]` với mọi `k < n/2`.

In [3]:
s = "ulaalu"; n = len(s)
print([(k, n - 1 - k, s[k], s[n - 1 - k]) for k in range(n // 2)])
a = "ulacfd"
print("cắt tại 3:", repr(a[:3]), repr(a[3:]), "| cắt tại 0:", repr(a[:0]), repr(a[0:]), "| cắt tại n:", repr(a[:6]), repr(a[6:]))

[(0, 5, 'u', 'u'), (1, 4, 'l', 'l'), (2, 3, 'a', 'a')]
cắt tại 3: 'ula' 'cfd' | cắt tại 0: '' 'ulacfd' | cắt tại n: 'ulacfd' ''


### 3.3 Kiểm tra palindrome trên một đoạn bằng hai con trỏ

Thay vì tạo chuỗi con mới (tốn bộ nhớ), dùng hai con trỏ `i` (trái) và `j` (phải) tiến vào giữa. Nếu gặp `s[i] != s[j]` thì không phải palindrome.

In [4]:
def is_pal_range(s, i, j):
    """s[i..j] (bao gồm cả hai đầu) có phải palindrome không. Đoạn rỗng (i > j) -> True."""
    while i < j:
        if s[i] != s[j]:
            return False
        i += 1; j -= 1
    return True

print(is_pal_range("xabay", 1, 3), is_pal_range("xabcy", 1, 3), is_pal_range("ab", 1, 0))

True False True


### 3.4 Hàm lồng và gọi đối xứng `f(a, b) or f(b, a)`

Đề có hai trường hợp đối xứng (đổi vai `a` và `b`). Mẹo: viết một hàm `check(x, y)` cho trường hợp `x_prefix + y_suffix`, rồi trả về `check(a, b) or check(b, a)`. Toán tử `or` **ngắt mạch**: nếu vế trái `True` thì không tính vế phải.

In [5]:
def f(x, y):
    print("  gọi f với", x, y)
    return x > y
print(f(3, 1) or f(1, 3))
print(f(1, 3) or f(3, 1))

  gọi f với 3 1
True
  gọi f với 1 3
  gọi f với 3 1
True


## 4. Ý tưởng nền tảng / trực giác

Liệt kê bằng tay **mọi** điểm cắt cho **Ví dụ 3** `a = "ulacfd"`, `b = "jizalu"` (`n = 6`, có 7 điểm cắt `0..6`):

In [6]:
a, b, _ = examples[2]
n = len(a)
for i in range(n + 1):
    s1, s2 = a[:i] + b[i:], b[:i] + a[i:]
    print(f"i={i}: a[:i]+b[i:]={s1!r:<9} {'PAL' if s1 == s1[::-1] else '   '} | b[:i]+a[i:]={s2!r:<9} {'PAL' if s2 == s2[::-1] else ''}")

i=0: a[:i]+b[i:]='jizalu'      | b[:i]+a[i:]='ulacfd'  
i=1: a[:i]+b[i:]='uizalu'      | b[:i]+a[i:]='jlacfd'  
i=2: a[:i]+b[i:]='ulzalu'      | b[:i]+a[i:]='jiacfd'  
i=3: a[:i]+b[i:]='ulaalu'  PAL | b[:i]+a[i:]='jizcfd'  
i=4: a[:i]+b[i:]='ulaclu'      | b[:i]+a[i:]='jizafd'  
i=5: a[:i]+b[i:]='ulacfu'      | b[:i]+a[i:]='jizald'  
i=6: a[:i]+b[i:]='ulacfd'      | b[:i]+a[i:]='jizalu'  


**Quan sát cấu trúc** của `s = a[:i] + b[i:]` khi `i ≤ n/2` (tiền tố ngắn):
- Vị trí `k < i` lấy từ `a`; vị trí gương `n−1−k ≥ n−i ≥ i` lấy từ `b` ⇒ cần **`a[k] == b[n−1−k]`** — đây là **vỏ ngoài**.
- Phần còn lại ở giữa, chỉ số `i..n−1−i`, nằm trọn trong `b` ⇒ cần **`b[i..n−1−i]` là palindrome** — đây là **lõi**.

Khi `i > n/2` thì tương tự nhưng lõi nằm trong `a`. Tóm lại: **vỏ ngoài** luôn là các cặp `(a[k], b[n−1−k])` với `k` từ 0 đến `L−1`, **lõi** là đoạn giữa `[L, n−1−L]` của **`a` hoặc `b`**.

Ở Ví dụ 3: `a[0]='u'=b[5]`, `a[1]='l'=b[4]`, `a[2]='a'=b[3]` ⇒ vỏ dài 3 = nửa chuỗi, lõi rỗng ⇒ palindrome.

## 5. Cách 1 — Brute Force / baseline: thử mọi điểm cắt

**Ý tưởng:** với mỗi `i` từ `0` đến `n` (đủ `n + 1` điểm cắt, gồm cả hai đầu), tạo hai chuỗi ghép và kiểm tra palindrome.

In [7]:
def check_bruteforce(a, b):
    n = len(a)
    for i in range(n + 1):
        s1 = a[:i] + b[i:]
        s2 = b[:i] + a[i:]
        if s1 == s1[::-1] or s2 == s2[::-1]:
            return True
    return False

for a, b, exp in examples:
    got = check_bruteforce(a, b)
    print(got, "kỳ vọng", exp); assert got == exp

True kỳ vọng True
False kỳ vọng False
True kỳ vọng True


**Độ phức tạp:** `n + 1` điểm cắt × (tạo chuỗi + đảo + so sánh `O(n)`) ⇒ `O(n²)` thời gian, `O(n)` bộ nhớ cho chuỗi tạm. Với `n = 10⁵` là `~10¹⁰` phép — **quá chậm**.

**Nút thắt:** mỗi điểm cắt ta kiểm tra lại toàn bộ chuỗi từ đầu, dù các điểm cắt kề nhau có vỏ ngoài gần như giống hệt nhau. Thử đo:

In [8]:
rng = random.Random(0)
for n in [500, 1000, 2000]:
    a = "".join(rng.choice("ab") for _ in range(n)); b = "".join(rng.choice("ab") for _ in range(n))
    a = "c" + a[1:]; b = b[:-1] + "d"      # đảm bảo không khớp sớm để phải thử hết điểm cắt
    t = time.perf_counter(); check_bruteforce(a, b); dt = time.perf_counter() - t
    print(f"n={n}: {dt*1000:.1f} ms")

n=500: 0.3 ms
n=1000: 1.2 ms
n=2000: 4.4 ms


Thời gian tăng khoảng **gấp 4** khi `n` gấp đôi — đúng dáng `O(n²)`. Với `n = 10⁵` (gấp 50 lần `n = 2000`) sẽ chậm hơn ~2500 lần ⇒ cỡ hơn chục giây (chưa kể chuỗi dài làm tăng chi phí cấp phát) ⇒ TLE.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (vỏ + lõi):** mọi chuỗi ghép palindrome dạng `a_prefix + b_suffix` đều có dạng: `L` cặp vỏ ngoài `a[k] == b[n−1−k]` (`k < L`) và lõi `a[L..n−1−L]` **hoặc** `b[L..n−1−L]` là palindrome.

**Insight 2 (palindrome đồng tâm):** các lõi `s[L..n−1−L]` với `L` khác nhau đều có **cùng tâm** (tâm của chuỗi). Nếu `s[L..n−1−L]` là palindrome thì mọi đoạn **bên trong** `s[L'..n−1−L']` với `L' ≥ L` cũng là palindrome (bóc dần từng cặp ở hai đầu).

**Hệ quả — tham lam:** hãy khớp vỏ **dài nhất có thể**: `i = 0, j = n−1`, tiến vào trong chừng nào `a[i] == b[j]`. Khi dừng:
- nếu `i ≥ j`: vỏ đã phủ hết ⇒ `True`;
- ngược lại kiểm tra lõi `a[i..j]` **hoặc** `b[i..j]` là palindrome.

**Quy tắc "nếu… thì…":** nếu có **một** cách tách hợp lệ với vỏ dài `L` thì cách tách với vỏ dài nhất `L* ≥ L` cũng hợp lệ (vỏ dài hơn vẫn khớp theo định nghĩa của `L*`; lõi ngắn hơn nằm trong lõi cũ nên vẫn palindrome). Vì vậy chỉ cần thử **đúng một** `L = L*`.

Làm hai lần: `check(a, b)` cho `a_prefix + b_suffix` và `check(b, a)` cho `b_prefix + a_suffix`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam hai con trỏ (khớp vỏ dài nhất rồi kiểm tra lõi)

**Giải thích từng đoạn:**
1. `i = 0`, `j = n − 1`. Trong khi `i < j` và `x[i] == y[j]`: `i += 1`, `j -= 1`.
2. Nếu `i ≥ j` ⇒ `True`.
3. Ngược lại trả về `is_pal_range(x, i, j) or is_pal_range(y, i, j)`.
4. Kết quả cuối: `check(a, b) or check(b, a)`.

Code demo (có in vỏ, lõi):

In [9]:
def check_demo(x, y):
    i, j = 0, len(y) - 1
    while i < j and x[i] == y[j]:
        i += 1; j -= 1
    if i >= j:
        print(f"   vỏ phủ hết (i={i}, j={j}) -> True")
        return True
    cx, cy = x[i:j + 1], y[i:j + 1]
    ok = cx == cx[::-1] or cy == cy[::-1]
    print(f"   vỏ dài {i}: {x[:i]!r}~{y[j+1:]!r} | lõi x={cx!r} ({cx == cx[::-1]}), lõi y={cy!r} ({cy == cy[::-1]})")
    return ok

for a, b, exp in examples:
    print(f"a={a!r}, b={b!r}")
    print("  check(a,b):"); r1 = check_demo(a, b)
    print("  check(b,a):"); r2 = check_demo(b, a)
    print("  =>", r1 or r2, "kỳ vọng", exp)

a='x', b='y'
  check(a,b):
   vỏ phủ hết (i=0, j=0) -> True
  check(b,a):
   vỏ phủ hết (i=0, j=0) -> True
  => True kỳ vọng True
a='xbdef', b='xecab'
  check(a,b):
   vỏ dài 0: ''~'' | lõi x='xbdef' (False), lõi y='xecab' (False)
  check(b,a):
   vỏ dài 0: ''~'' | lõi x='xecab' (False), lõi y='xbdef' (False)
  => False kỳ vọng False
a='ulacfd', b='jizalu'
  check(a,b):
   vỏ phủ hết (i=3, j=2) -> True
  check(b,a):
   vỏ dài 0: ''~'' | lõi x='jizalu' (False), lõi y='ulacfd' (False)
  => True kỳ vọng True


**Code nộp LeetCode (Cách 2):** xem mục 9 (đây là bản được chọn để nộp).

**Độ phức tạp Cách 2:** con trỏ đi tối đa `n/2` bước, kiểm tra lõi tối đa `O(n)`, làm 2 lần ⇒ Time `O(n)`, Space `O(1)` nếu kiểm tra lõi bằng hai con trỏ (hoặc `O(n)` nếu cắt chuỗi `x[i:j+1]`).

### 7.2 Cách 3 — Mở rộng từ tâm (tính lõi palindrome lớn nhất trước)

Đảo ngược thứ tự suy nghĩ: thay vì đi từ **ngoài vào**, ta đi từ **trong ra**.

**Giải thích từng đoạn:**
1. Với mỗi chuỗi `s ∈ {a, b}`, mở rộng từ tâm để tìm `L_s` = **độ dài vỏ nhỏ nhất** sao cho `s[L_s..n−1−L_s]` là palindrome (lõi palindrome đồng tâm dài nhất).
2. Tính `P_ab` = số cặp đầu tiên liên tiếp khớp `a[k] == b[n−1−k]`, và `P_ba` = số cặp khớp `b[k] == a[n−1−k]`.
3. `a_prefix + b_suffix` khả thi ⇔ `P_ab ≥ min(L_a, L_b)` (lõi lấy từ `a` hoặc `b`, vỏ phải phủ được phần ngoài lõi). Tương tự cho `b_prefix + a_suffix` với `P_ba`.

In [10]:
def core_outer_len(s):
    """L nhỏ nhất sao cho s[L .. n-1-L] là palindrome (mở rộng từ tâm)."""
    n = len(s)
    i, j = (n - 1) // 2, n // 2          # n lẻ: i == j (tâm 1 ký tự); n chẵn: tâm giữa hai ký tự
    while i >= 0 and s[i] == s[j]:
        i -= 1; j += 1
    return i + 1

def match_len(x, y):
    """Số k liên tiếp từ 0 sao cho x[k] == y[n-1-k] (dừng ở n//2 là đủ)."""
    n, k = len(x), 0
    while k < n // 2 and x[k] == y[n - 1 - k]:
        k += 1
    return k

for s in ["abcba", "xabbay", "xyz", "ulacfd", "jizalu", "a"]:
    L = core_outer_len(s)
    print(f"{s!r}: L={L}, lõi={s[L:len(s)-L]!r}")

'abcba': L=0, lõi='abcba'
'xabbay': L=1, lõi='abba'
'xyz': L=1, lõi='y'
'ulacfd': L=3, lõi=''
'jizalu': L=3, lõi=''
'a': L=0, lõi='a'


In [11]:
def check_center(a, b):
    La, Lb = core_outer_len(a), core_outer_len(b)
    need = min(La, Lb)
    return match_len(a, b) >= need or match_len(b, a) >= need

for a, b, exp in examples:
    print(f"a={a!r}: L_a={core_outer_len(a)} | b={b!r}: L_b={core_outer_len(b)} | P_ab={match_len(a,b)}, P_ba={match_len(b,a)} -> {check_center(a, b)} (kỳ vọng {exp})")
    assert check_center(a, b) == exp

a='x': L_a=0 | b='y': L_b=0 | P_ab=0, P_ba=0 -> True (kỳ vọng True)
a='xbdef': L_a=2 | b='xecab': L_b=2 | P_ab=0, P_ba=0 -> False (kỳ vọng False)
a='ulacfd': L_a=3 | b='jizalu': L_b=3 | P_ab=3, P_ba=0 -> True (kỳ vọng True)


**Vì sao dùng `min(L_a, L_b)`:** nếu vỏ khớp được `P` cặp và `P ≥ L_a` thì chọn lõi từ `a` — lõi `a[P..n−1−P]` nằm bên trong `a[L_a..n−1−L_a]` nên là palindrome. Tương tự với `b`. Vậy chỉ cần `P ≥ L_a` **hoặc** `P ≥ L_b` ⇔ `P ≥ min(L_a, L_b)`.

**Code nộp LeetCode (Cách 3):**

In [12]:
class SolutionCenter:
    def checkPalindromeFormation(self, a: str, b: str) -> bool:
        n = len(a)

        def outer(s: str) -> int:
            i, j = (n - 1) // 2, n // 2
            while i >= 0 and s[i] == s[j]:
                i -= 1
                j += 1
            return i + 1

        def match(x: str, y: str) -> int:
            k = 0
            while k < n // 2 and x[k] == y[n - 1 - k]:
                k += 1
            return k

        need = min(outer(a), outer(b))
        return match(a, b) >= need or match(b, a) >= need

print([SolutionCenter().checkPalindromeFormation(a, b) for a, b, _ in examples])

[True, False, True]


**Độ phức tạp Cách 3:** mỗi lần mở rộng/khớp tối đa `n/2` bước ⇒ Time `O(n)`, Space `O(1)`.

### 7.3 Cách 4 — Tham lam viết gọn bằng slicing (phiên bản "Pythonic")

Cùng ý tưởng Cách 2 nhưng kiểm tra lõi bằng `mid == mid[::-1]` — chạy nhanh vì slicing và so sánh chuỗi được cài bằng C, đổi lại tốn `O(n)` bộ nhớ tạm.

In [13]:
class SolutionSlice:
    def checkPalindromeFormation(self, a: str, b: str) -> bool:
        def check(x: str, y: str) -> bool:
            i, j = 0, len(x) - 1
            while i < j and x[i] == y[j]:
                i += 1
                j -= 1
            mx, my = x[i:j + 1], y[i:j + 1]
            return mx == mx[::-1] or my == my[::-1]
        return check(a, b) or check(b, a)

print([SolutionSlice().checkPalindromeFormation(a, b) for a, b, _ in examples])

[True, False, True]


Khi `i ≥ j`, `x[i:j+1]` là chuỗi rỗng (hoặc 1 ký tự) nên tự động là palindrome — không cần nhánh `if i >= j` riêng.

**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(n)` (chuỗi lõi tạm).

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 2**: `a = "xbdef"`, `b = "xecab"` (`n = 5`).

`check(a, b)`:

| bước | i | j | a[i] | b[j] | khớp? |
|---|---|---|---|---|---|
| 1 | 0 | 4 | x | b | không → dừng |

Lõi `a[0..4] = "xbdef"` (không), `b[0..4] = "xecab"` (không) ⇒ `False`.

`check(b, a)`:

| bước | i | j | b[i] | a[j] | khớp? |
|---|---|---|---|---|---|
| 1 | 0 | 4 | x | f | không → dừng |

Lõi `"xecab"`, `"xbdef"` đều không ⇒ `False`. Kết quả `False`. Hàm `trace_check` in ra từng bước:

In [14]:
def trace_check(x, y, label):
    print(f"--- {label}: x={x!r}, y={y!r}")
    i, j = 0, len(y) - 1
    step = 0
    while i < j:
        step += 1
        ok = x[i] == y[j]
        print(f"bước {step}: i={i}, j={j}, x[i]={x[i]!r}, y[j]={y[j]!r} -> {'khớp, tiến vào' if ok else 'KHÔNG khớp, dừng'}")
        if not ok:
            break
        i += 1; j -= 1
    if i >= j:
        print(f"i={i} >= j={j}: vỏ phủ hết -> True"); return True
    px, py = is_pal_range(x, i, j), is_pal_range(y, i, j)
    print(f"lõi x[{i}..{j}]={x[i:j+1]!r} palindrome? {px} | lõi y[{i}..{j}]={y[i:j+1]!r} palindrome? {py}")
    return px or py

def trace_solution(a, b):
    r = trace_check(a, b, "a_prefix + b_suffix") or trace_check(b, a, "b_prefix + a_suffix")
    print("KẾT QUẢ:", r)
    return r

assert trace_solution("xbdef", "xecab") is False

--- a_prefix + b_suffix: x='xbdef', y='xecab'
bước 1: i=0, j=4, x[i]='x', y[j]='b' -> KHÔNG khớp, dừng
lõi x[0..4]='xbdef' palindrome? False | lõi y[0..4]='xecab' palindrome? False
--- b_prefix + a_suffix: x='xecab', y='xbdef'
bước 1: i=0, j=4, x[i]='x', y[j]='f' -> KHÔNG khớp, dừng
lõi x[0..4]='xecab' palindrome? False | lõi y[0..4]='xbdef' palindrome? False
KẾT QUẢ: False


Trace trên **Ví dụ 3** (vỏ phủ hết) và trên ví dụ tự dựng `a = "abxyz"`, `b = "qcdba"` (ví dụ tự dựng — kết quả được đối chiếu với brute force ngay trong cell; nên đối chiếu lại đề gốc):

In [15]:
assert trace_solution("ulacfd", "jizalu") is True
print()
assert trace_solution("abxyz", "qcdba") == check_bruteforce("abxyz", "qcdba")

--- a_prefix + b_suffix: x='ulacfd', y='jizalu'
bước 1: i=0, j=5, x[i]='u', y[j]='u' -> khớp, tiến vào
bước 2: i=1, j=4, x[i]='l', y[j]='l' -> khớp, tiến vào
bước 3: i=2, j=3, x[i]='a', y[j]='a' -> khớp, tiến vào
i=3 >= j=2: vỏ phủ hết -> True
KẾT QUẢ: True

--- a_prefix + b_suffix: x='abxyz', y='qcdba'
bước 1: i=0, j=4, x[i]='a', y[j]='a' -> khớp, tiến vào
bước 2: i=1, j=3, x[i]='b', y[j]='b' -> khớp, tiến vào
i=2 >= j=2: vỏ phủ hết -> True
KẾT QUẢ: True


Ở ví dụ tự dựng này: vỏ `a[0]='a'=b[4]`, `a[1]='b'=b[3]` ⇒ `i = 2 = j` ⇒ vỏ đã chạm tâm ⇒ `True`, chuỗi thu được `a[:2] + b[2:] = "ab" + "dba" = "abdba"`. Ví dụ tự dựng tiếp theo buộc thuật toán phải **kiểm tra lõi thật** — và lõi palindrome nằm trong `b` (đối chiếu brute force trong cell; nên đối chiếu lại đề gốc):

In [16]:
a_, b_ = "axyzq", "pcdca"      # ví dụ tự dựng: vỏ dài 1, lõi b[1..3] = "cdc"
assert trace_solution(a_, b_) is True and check_bruteforce(a_, b_) is True
print("Chuỗi thu được:", a_[:1] + b_[1:])

--- a_prefix + b_suffix: x='axyzq', y='pcdca'
bước 1: i=0, j=4, x[i]='a', y[j]='a' -> khớp, tiến vào
bước 2: i=1, j=3, x[i]='x', y[j]='c' -> KHÔNG khớp, dừng
lõi x[1..3]='xyz' palindrome? False | lõi y[1..3]='cdc' palindrome? True
KẾT QUẢ: True
Chuỗi thu được: acdca


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — tham lam hai con trỏ, kiểm tra lõi bằng hai con trỏ, `O(n)` thời gian, `O(1)` bộ nhớ.

In [17]:
class Solution:
    def checkPalindromeFormation(self, a: str, b: str) -> bool:
        def is_pal(s: str, i: int, j: int) -> bool:
            while i < j:
                if s[i] != s[j]:
                    return False
                i += 1
                j -= 1
            return True

        def check(x: str, y: str) -> bool:
            i, j = 0, len(x) - 1
            while i < j and x[i] == y[j]:
                i += 1
                j -= 1
            return i >= j or is_pal(x, i, j) or is_pal(y, i, j)

        return check(a, b) or check(b, a)

for a, b, exp in examples:
    got = Solution().checkPalindromeFormation(a, b)
    print(got, exp); assert got == exp

True True
False False
True True


## 10. Vì sao thuật toán đúng

Xét trường hợp `x_prefix + y_suffix` (trường hợp còn lại đối xứng). Gọi `L*` = độ dài vỏ mà vòng `while` dừng lại (số cặp đầu liên tiếp có `x[k] == y[n−1−k]`, tối đa đến khi hai con trỏ gặp nhau).

**(⇐) Thuật toán trả `True` thì có cách cắt.** Nếu `i ≥ j`: chọn điểm cắt `n/2` (làm tròn tùy ý) — mọi cặp gương khớp ⇒ palindrome. Nếu lõi `x[L*..n−1−L*]` palindrome: cắt tại `i = n − L*` ⇒ chuỗi `x[:n−L*] + y[n−L*:]` có vỏ `x[k]` vs `y[n−1−k]` (`k < L*`) khớp và lõi từ `x` palindrome. Nếu lõi `y[...]` palindrome: cắt tại `i = L*`.

**(⇒) Có cách cắt thì thuật toán trả `True`.** Giả sử điểm cắt `i` tạo palindrome. Như mục 4, có `L ≤ n/2` sao cho `x[k] == y[n−1−k]` với `k < L` và lõi `z[L..n−1−L]` palindrome với `z ∈ {x, y}`. Theo định nghĩa của `L*` (khớp dài nhất), `L* ≥ L` (hoặc hai con trỏ đã gặp nhau ⇒ `True`). Lõi `z[L*..n−1−L*]` nằm **đồng tâm, bên trong** `z[L..n−1−L]` ⇒ cũng palindrome ⇒ thuật toán trả `True`. ∎

**Cách 3** dùng đúng hai bổ đề trên, chỉ khác thứ tự tính: `L_z` là lõi palindrome đồng tâm lớn nhất, điều kiện khả thi là vỏ khớp được ít nhất `L_z` cặp.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(a) = len(b) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi điểm cắt | `O(n²)` | `O(n)` chuỗi tạm |
| 2. Tham lam + hai con trỏ | `O(n)` (≤ `2·(n/2 + n)` phép so sánh) | `O(1)` |
| 3. Mở rộng từ tâm | `O(n)` | `O(1)` |
| 4. Tham lam + slicing | `O(n)` | `O(n)` |

Không có khác biệt average/worst đáng kể cho Cách 2–4: tệ nhất mỗi con trỏ đi hết nửa chuỗi. Đo trên `n = 10⁵`:

In [18]:
rng = random.Random(7)
n = 10 ** 5
half = "".join(rng.choice("ab") for _ in range(n // 2))
A = half + "".join(rng.choice("ab") for _ in range(n // 2))
B = "".join(rng.choice("ab") for _ in range(n // 2)) + half[::-1]     # vỏ khớp toàn bộ -> True
A2 = "c" + A[1:]                                                       # phá khớp ngay đầu
for name, S in [("Cách 2", Solution()), ("Cách 3", SolutionCenter()), ("Cách 4", SolutionSlice())]:
    t = time.perf_counter(); r1 = S.checkPalindromeFormation(A, B); r2 = S.checkPalindromeFormation(A2, B); dt = time.perf_counter() - t
    print(f"{name}: {r1}, {r2}  ({dt*1000:.1f} ms cho 2 lần gọi)")

Cách 2: True, False  (1.9 ms cho 2 lần gọi)
Cách 3: True, False  (2.8 ms cho 2 lần gọi)
Cách 4: True, False  (2.1 ms cho 2 lần gọi)


## 12. Bộ test

Gồm: ví dụ đề, edge case (`n = 1`, một chuỗi đã là palindrome, chỉ cách ghép thứ hai mới đúng, lõi nằm trong `b`), và đối chiếu mọi cách với brute force trên nhiều cặp chuỗi ngẫu nhiên có seed cố định (bảng chữ nhỏ để có nhiều trường hợp `True`).

In [19]:
ALL = {
    "brute": check_bruteforce,
    "center": check_center,
    "submit": lambda a, b: Solution().checkPalindromeFormation(a, b),
    "SolCenter": lambda a, b: SolutionCenter().checkPalindromeFormation(a, b),
    "SolSlice": lambda a, b: SolutionSlice().checkPalindromeFormation(a, b),
}
def run_all(a, b):
    res = {k: f(a, b) for k, f in ALL.items()}
    assert len(set(res.values())) == 1, (a, b, res)
    return next(iter(res.values()))

for a, b, exp in examples:
    assert run_all(a, b) == exp
edge = [
    ("a", "a", True), ("a", "b", True),                 # n = 1 luôn True
    ("ab", "cd", False), ("ab", "ca", True),            # "a"+"a" = "aa"
    ("abc", "xyz", False), ("aba", "xyz", True),        # a đã là palindrome
    ("jizalu", "ulacfd", True),                         # đổi vai Ví dụ 3: chỉ b_prefix + a_suffix đúng
    ("axyzq", "pcdca", True),                           # lõi nằm trong b (tự dựng)
    ("pqcdcz", "zxwyxp", False),
]
for a, b, exp in edge:
    assert check_bruteforce(a, b) == exp, (a, b)
    assert run_all(a, b) == exp
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [20]:
rng = random.Random(1616)
stat = {True: 0, False: 0}
for _ in range(3000):
    n = rng.randint(1, 12)
    alpha = rng.choice(["ab", "abc", "abcd"])
    a = "".join(rng.choice(alpha) for _ in range(n))
    b = "".join(rng.choice(alpha) for _ in range(n))
    stat[run_all(a, b)] += 1
print("3000 cặp ngẫu nhiên OK | phân bố:", stat)

# Ngẫu nhiên có cấu trúc: cố ý tạo vỏ khớp + lõi palindrome rồi làm nhiễu
for _ in range(1000):
    L = rng.randint(0, 6); m = rng.randint(0, 6)
    outer = "".join(rng.choice("ab") for _ in range(L))
    core_half = "".join(rng.choice("ab") for _ in range(m // 2))
    core = core_half + (rng.choice("ab") if m % 2 else "") + core_half[::-1]
    filler = "".join(rng.choice("ab") for _ in range(len(core)))
    a = outer + core + "".join(rng.choice("ab") for _ in range(L))
    b = "".join(rng.choice("ab") for _ in range(L)) + filler + outer[::-1]
    if not a:
        continue
    if rng.random() < 0.3:
        k = rng.randrange(len(a)); a = a[:k] + "c" + a[k + 1:]
    run_all(a, b)
print("1000 cặp có cấu trúc OK")

3000 cặp ngẫu nhiên OK | phân bố: {True: 1178, False: 1822}
1000 cặp có cấu trúc OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra `a_prefix + b_suffix`, quên `b_prefix + a_suffix`

In [21]:
def wrong_one_side(a, b):
    i, j = 0, len(a) - 1
    while i < j and a[i] == b[j]:
        i += 1; j -= 1
    return i >= j or is_pal_range(a, i, j) or is_pal_range(b, i, j)

a, b = "jizalu", "ulacfd"
print("sai :", wrong_one_side(a, b), "| đúng:", Solution().checkPalindromeFormation(a, b))

sai : False | đúng: True


**Vì sao sai:** palindrome duy nhất ở đây là `b_prefix + a_suffix = "ula" + "alu"`. **Sửa:** `return check(a, b) or check(b, a)`.

### Lỗi 2 — Sau khi vỏ dừng, chỉ kiểm tra lõi của **một** chuỗi

In [22]:
def wrong_core_one(a, b):
    def check(x, y):
        i, j = 0, len(x) - 1
        while i < j and x[i] == y[j]:
            i += 1; j -= 1
        return i >= j or is_pal_range(x, i, j)        # SAI: quên lõi của y
    return check(a, b) or check(b, a)

a, b = "axyzq", "pcdca"
print("sai :", wrong_core_one(a, b), "| đúng:", Solution().checkPalindromeFormation(a, b), "| brute:", check_bruteforce(a, b))

sai : False | đúng: True | brute: True


**Vì sao sai:** lõi có thể lấy từ chuỗi cung cấp **hậu tố** (khi điểm cắt ở nửa đầu) hoặc chuỗi cung cấp **tiền tố** (điểm cắt ở nửa sau). Ở đây lõi `"cdc"` nằm trong `b`: `"a" + "cdca" = "acdca"`. **Sửa:** `is_pal(x, i, j) or is_pal(y, i, j)`.

### Lỗi 3 — So sánh `a[i]` với `b[i]` thay vì với vị trí gương `b[n−1−i]`

In [23]:
def wrong_mirror(a, b):
    def check(x, y):
        i, j = 0, len(x) - 1
        while i < j and x[i] == y[i]:                 # SAI: phải là y[j]
            i += 1; j -= 1
        return i >= j or is_pal_range(x, i, j) or is_pal_range(y, i, j)
    return check(a, b) or check(b, a)

a, b, _ = examples[2]
print("sai :", wrong_mirror(a, b), "| đúng:", Solution().checkPalindromeFormation(a, b))

sai : False | đúng: True


**Vì sao sai:** palindrome so ký tự ở vị trí `k` với vị trí **đối xứng** `n−1−k`. Trong `a[:i] + b[i:]`, ký tự đầu là `a[0]`, ký tự cuối là `b[n−1]`. **Sửa:** so `x[i]` với `y[j]` khi `j = n−1−i`.

### Lỗi 4 — Brute force bỏ sót điểm cắt ở hai đầu (`range(1, n)`)

In [24]:
def wrong_range(a, b):
    for i in range(1, len(a)):                         # SAI: thiếu i = 0 và i = n
        s1, s2 = a[:i] + b[i:], b[:i] + a[i:]
        if s1 == s1[::-1] or s2 == s2[::-1]:
            return True
    return False

a, b, _ = examples[0]
print("sai :", wrong_range(a, b), "| đúng:", check_bruteforce(a, b))
print("thêm:", wrong_range("aba", "xyz"), "| đúng:", check_bruteforce("aba", "xyz"))

sai : False | đúng: True
thêm: False | đúng: True


**Vì sao sai:** đề cho phép tiền tố/hậu tố **rỗng**. Với `n = 1` không còn điểm cắt nào; với `a = "aba"` thì cách duy nhất là lấy nguyên `a`. **Sửa:** `range(n + 1)`.

### Lỗi 5 — Tham lam nhưng "thử lùi" vỏ ngắn hơn, làm `O(n²)` không cần thiết

Một số bạn không tin tham lam nên sau khi vỏ dừng ở `L*`, thử lại mọi `L ≤ L*` và kiểm tra lõi cho từng `L`. Kết quả **đúng** nhưng chậm `O(n²)` trong trường hợp tệ — và hoàn toàn thừa vì bổ đề đồng tâm.

In [25]:
def slow_try_all_L(a, b):
    n = len(a)
    def check(x, y):
        i, j = 0, n - 1
        while i < j and x[i] == y[j]:
            i += 1; j -= 1
        if i >= j:
            return True
        for L in range(i, -1, -1):                     # thừa: thử mọi vỏ ngắn hơn
            if is_pal_range(x, L, n - 1 - L) or is_pal_range(y, L, n - 1 - L):
                return True
        return False
    return check(a, b) or check(b, a)

h = 1500                                              # n = 2h + 2
a = "a" * h + "xy" + "a" * h                          # vỏ ngoài của a đối xứng, tâm "xy" thì không
b = "z" * h + "uv" + "a" * h
for f in (slow_try_all_L, Solution().checkPalindromeFormation):
    t = time.perf_counter(); r = f(a, b); dt = time.perf_counter() - t
    print(f"{getattr(f, '__name__', 'Solution')}: {r} ({dt*1000:.1f} ms)")

slow_try_all_L: False (47.8 ms)
checkPalindromeFormation: False (0.1 ms)


**Vì sao không cần:** nếu lõi ứng với vỏ `L*` không phải palindrome thì mọi lõi **bao ngoài** nó (vỏ ngắn hơn) cũng không phải palindrome (lõi ngoài chứa lõi trong ở tâm). **Sửa:** chỉ kiểm tra đúng một lõi ứng với `L*`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Ghép tiền tố chuỗi này với hậu tố chuỗi kia thành palindrome" ⇒ nghĩ ngay **vỏ ngoài + lõi giữa**.
- Palindrome + "thử mọi điểm cắt" ⇒ thường giảm được về `O(n)` nhờ **hai con trỏ từ hai đầu** hoặc **mở rộng từ tâm**.
- Bài có hai vai đối xứng ⇒ viết `check(x, y)` một lần, gọi `check(a, b) or check(b, a)`.
- Tính chất "palindrome đồng tâm lồng nhau" là chìa khóa để chứng minh tham lam (gặp lại ở 680 Valid Palindrome II, 5 Longest Palindromic Substring).
- Luôn test điểm cắt ở hai đầu (`i = 0`, `i = n`) và `n = 1`.

## 15. Biến thể của bài

1. **Trả về điểm cắt** (và cách ghép) thay vì `True/False` — Bài tập 4.
2. **Đếm số điểm cắt** tạo được palindrome — Bài tập 5 (làm được `O(n)` bằng `P_ab, P_ba, L_a, L_b`).
3. **Hai chuỗi khác độ dài**: điểm cắt độc lập trên mỗi chuỗi ⇒ bài khó hơn hẳn (cần Manacher/hashing).
4. **680. Valid Palindrome II** — được xóa tối đa một ký tự: cũng là hai con trỏ, khi lệch thì thử bỏ trái hoặc bỏ phải và kiểm tra lõi.
5. **125 / 234 / 5 / 647**: các bài palindrome cơ bản (hai con trỏ, mở rộng từ tâm).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Thử `n+1` điểm cắt, ghép và đảo chuỗi | `O(n²)` | `O(n)` | Kiểm chứng, `n` nhỏ |
| 2. Tham lam hai con trỏ | Khớp vỏ dài nhất, kiểm lõi của `x` hoặc `y` | `O(n)` | `O(1)` | **Bản nộp khuyên dùng** |
| 3. Mở rộng từ tâm | Tính lõi palindrome lớn nhất, so với độ dài vỏ khớp | `O(n)` | `O(1)` | Khi cần thêm thông tin (đếm điểm cắt) |
| 4. Tham lam + slicing | Như 2, lõi kiểm bằng `s == s[::-1]` | `O(n)` | `O(n)` | Code ngắn, nhanh trong Python |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `a="abc", b="cba"`; (b) `a="ab", b="ab"`; (c) `a="abda", b="acmc"`; (d) `a="pvhmupgqeltozftlmfjjde", b="yjgpzbezspnnpszebzmhvp"`.

**Bài 2 (tính tay):** Với `a = "abxef"`, `b = "feyba"`, trace Cách 2 cho `check(a, b)`: liệt kê `i, j` qua từng vòng, cho biết vỏ dài bao nhiêu và chuỗi palindrome thu được.

**Bài 3 (điền TODO):** Hoàn thiện `core_outer_len(s)` (mở rộng từ tâm) trả về `L` nhỏ nhất để `s[L..n−1−L]` là palindrome.

**Bài 4 (tự viết lại):** Viết `find_split(a, b)` trả về `(i, kiểu)` với `kiểu ∈ {"ab", "ba"}` là **một** điểm cắt hợp lệ (`"ab"` nghĩa là `a[:i] + b[i:]`), hoặc `None` nếu không có. Phải chạy `O(n)`.

**Bài 5 (mini challenge):** Viết `count_splits(a, b)` đếm số `i ∈ [0, n]` sao cho **ít nhất một** trong hai chuỗi ghép tại `i` là palindrome, trong `O(n)`.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def core_outer_len_todo(s):
    n = len(s)
    i, j = (n - 1) // 2, n // 2
    # TODO: trong khi i >= 0 và s[i] == s[j] thì mở rộng i, j ra hai phía
    return i + 1

In [27]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def find_split(a, b):
    # TODO: chạy tham lam cho (a, b) và (b, a); lõi từ x -> cắt tại n - L; lõi từ y -> cắt tại L
    pass

def count_splits(a, b):
    # TODO: với i <= n/2: cần P >= i và i >= L(lõi chuỗi hậu tố); với i > n/2: t = n - i, cần P >= t và t >= L(lõi chuỗi tiền tố)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: thử tham lam hai lần. Mức 2: nhớ cả lõi của hai chuỗi. Mức 3: (a) `True` (`a[0]='a'=b[2]`, hai con trỏ gặp nhau ⇒ ví dụ `"a" + "ba" = "aba"`); (b) `False` (`'a'` vs `'b'` lệch ngay ở cả hai chiều, lõi `"ab"` không đối xứng); (c) `False` — chiều `(a, b)`: `'a'` vs `'c'` lệch, lõi `"abda"`, `"acmc"` đều không; chiều `(b, a)`: `'a'='a'` khớp, rồi `'c'` vs `'d'` lệch, lõi `"cm"`, `"bd"` đều không; (d) `True` — vỏ khớp được vài cặp rồi lõi của một chuỗi là palindrome; hãy chạy `trace_solution` để xem chi tiết.

**Bài 2.** Mức 1: `i=0, j=4`: `'a'` vs `'a'`. Mức 2: `i=1, j=3`: `'b'` vs `'b'`; `i=2, j=2` dừng vì `i < j` sai. Mức 3: vỏ dài 2, `i ≥ j` ⇒ `True`; chuỗi `a[:2] + b[2:] = "ab" + "yba" = "abyba"`.

**Bài 3.** Mức 1: `while i >= 0 and s[i] == s[j]`. Mức 2: `i -= 1; j += 1`. Mức 3: chuỗi chẵn bắt đầu với lõi rỗng (`i = j − 1`), chuỗi lẻ bắt đầu với lõi 1 ký tự (`i == j`).

**Bài 4.** Mức 1: tham lam được `L*`. Mức 2: `i ≥ j` ⇒ cắt tại `L*`. Mức 3: lõi `x` palindrome ⇒ cắt tại `n − L*` (tiền tố `x` dài, chứa lõi); lõi `y` palindrome ⇒ cắt tại `L*`. Với `check(b, a)` thì kiểu là `"ba"`.

**Bài 5.** Mức 1: tính `P_ab, P_ba` bằng `match_len` (dừng ở `n//2` là đủ vì mọi vỏ cần xét đều dài `≤ n/2`) và `L_a, L_b` bằng `core_outer_len`. Mức 2: kiểu `"ab"` tại `i` (`2i ≤ n`): `P_ab ≥ i` và `i ≥ L_b`; (`2i > n`, `t = n − i`): `P_ab ≥ t` và `t ≥ L_a`. Mức 3: kiểu `"ba"` đổi vai: `P_ba`, lõi `a` khi `2i ≤ n`, lõi `b` khi `2i > n`. Đối chiếu bằng brute force.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2
b1 = [Solution().checkPalindromeFormation(x, y) for x, y in
      [("abc", "cba"), ("ab", "ab"), ("abda", "acmc"), ("pvhmupgqeltozftlmfjjde", "yjgpzbezspnnpszebzmhvp")]]
print("Bài 1:", b1)
assert b1 == [check_bruteforce(x, y) for x, y in
              [("abc", "cba"), ("ab", "ab"), ("abda", "acmc"), ("pvhmupgqeltozftlmfjjde", "yjgpzbezspnnpszebzmhvp")]]
assert b1 == [True, False, False, True]
print("Bài 2:"); assert trace_check("abxef", "feyba", "a_prefix + b_suffix") is True
print("   chuỗi:", "abxef"[:2] + "feyba"[2:])

Bài 1: [True, False, False, True]
Bài 2:
--- a_prefix + b_suffix: x='abxef', y='feyba'
bước 1: i=0, j=4, x[i]='a', y[j]='a' -> khớp, tiến vào
bước 2: i=1, j=3, x[i]='b', y[j]='b' -> khớp, tiến vào
i=2 >= j=2: vỏ phủ hết -> True
   chuỗi: abyba


In [29]:
# Bài 3, 4 — đáp án
def core_outer_len_todo(s):
    n = len(s)
    i, j = (n - 1) // 2, n // 2
    while i >= 0 and s[i] == s[j]:
        i -= 1; j += 1
    return i + 1

def find_split(a, b):
    n = len(a)
    for x, y, kind in ((a, b, "ab"), (b, a, "ba")):
        i, j = 0, n - 1
        while i < j and x[i] == y[j]:
            i += 1; j -= 1
        if i >= j:
            return (i, kind)
        if is_pal_range(x, i, j):
            return (n - i, kind)
        if is_pal_range(y, i, j):
            return (i, kind)
    return None

def valid_split(a, b, ans):
    if ans is None:
        return not check_bruteforce(a, b)
    i, kind = ans
    s = a[:i] + b[i:] if kind == "ab" else b[:i] + a[i:]
    return 0 <= i <= len(a) and s == s[::-1]

rng = random.Random(99)
for _ in range(2000):
    n = rng.randint(1, 10)
    s = "".join(rng.choice("abc") for _ in range(n))
    assert core_outer_len_todo(s) == min(L for L in range(n // 2 + 1) if s[L:n - L] == s[L:n - L][::-1])
    a = "".join(rng.choice("ab") for _ in range(n)); b = "".join(rng.choice("ab") for _ in range(n))
    assert valid_split(a, b, find_split(a, b)), (a, b)
print("Bài 3, 4: OK | ví dụ:", find_split("ulacfd", "jizalu"), find_split("axyzq", "pcdca"), find_split("xbdef", "xecab"))

Bài 3, 4: OK | ví dụ: (3, 'ab') (1, 'ab') None


In [30]:
# Bài 5 — đáp án O(n)
def count_splits(a, b):
    n = len(a)
    La, Lb = core_outer_len(a), core_outer_len(b)
    Pab, Pba = match_len(a, b), match_len(b, a)
    total = 0
    for i in range(n + 1):
        if 2 * i <= n:
            ok_ab = Pab >= i and i >= Lb          # a[:i] + b[i:], lõi trong b
            ok_ba = Pba >= i and i >= La          # b[:i] + a[i:], lõi trong a
        else:
            t = n - i
            ok_ab = Pab >= t and t >= La          # lõi trong a
            ok_ba = Pba >= t and t >= Lb          # lõi trong b
        total += ok_ab or ok_ba
    return total

def count_brute(a, b):
    n = len(a); c = 0
    for i in range(n + 1):
        s1, s2 = a[:i] + b[i:], b[:i] + a[i:]
        c += (s1 == s1[::-1]) or (s2 == s2[::-1])
    return c

rng = random.Random(5)
for _ in range(3000):
    n = rng.randint(1, 10)
    a = "".join(rng.choice("ab") for _ in range(n)); b = "".join(rng.choice("ab") for _ in range(n))
    assert count_splits(a, b) == count_brute(a, b), (a, b)
print("Bài 5: OK | Ví dụ 3:", count_splits("ulacfd", "jizalu"), "| 'aba','aba':", count_splits("aba", "aba"))

Bài 5: OK | Ví dụ 3: 1 | 'aba','aba': 4


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "vỏ ngoài tham lam + lõi palindrome":**

```python
def is_pal(s, i, j):
    while i < j:
        if s[i] != s[j]: return False
        i += 1; j -= 1
    return True

def check(x, y):                      # x_prefix + y_suffix
    i, j = 0, len(x) - 1
    while i < j and x[i] == y[j]:     # khớp vỏ dài nhất
        i += 1; j -= 1
    return i >= j or is_pal(x, i, j) or is_pal(y, i, j)

return check(a, b) or check(b, a)
```

**Checklist:**
- [ ] Gọi **cả hai** chiều `check(a, b)` và `check(b, a)`?
- [ ] So `x[i]` với `y[j]` (vị trí **gương**), không phải `y[i]`?
- [ ] Kiểm tra lõi của **cả** `x` và `y`?
- [ ] Trường hợp vỏ phủ hết (`i ≥ j`) trả `True`?
- [ ] Không thử lại các vỏ ngắn hơn (thừa, `O(n²)`)?
- [ ] Đã test `n = 1`, chuỗi đã là palindrome, ví dụ chỉ đúng khi đổi vai, lõi nằm ở chuỗi thứ hai, và đối chiếu brute force ngẫu nhiên?